In [2]:
from pymongo import MongoClient
import pandas as pd

df = pd.read_csv('../TP_jour01/data/processed/commandes_ecommerce.csv')
print("Lignes lues :", len(df))

df = df.drop_duplicates()

for c in ['quantity', 'unit_price', 'line_total', 'discounted_line_total']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['brand'] = df['brand'].fillna('inconnu')

db = MongoClient()['commerce']
col = db['raw_commandes']
col.delete_many({})
col.insert_many(df.to_dict('records'))

print("Documents insérés :", col.count_documents({}))
print("Paniers distincts :", len(col.distinct('cart_id')))

Lignes lues : 800
Documents insérés : 800
Paniers distincts : 208


In [3]:
import requests

ressources = ["products", "carts", "users", "posts", "comments", "recipes", "quotes", "todos"]

for r in ressources:
    reponse = requests.get(f"https://dummyjson.com/{r}?limit=0")
    documents = reponse.json()[r]
    col = db["raw_" + r]
    col.delete_many({})
    col.insert_many(documents)
    print(r, ":", col.count_documents({}))

products : 194
carts : 208
users : 208
posts : 251
comments : 340
recipes : 50
quotes : 1454
todos : 254


In [4]:
import random

random.seed(2026)

paiements = []
for panier in db["raw_carts"].find():
    statut = random.choices(["PAID", "REFUSED", "PENDING"], weights=[0.7, 0.2, 0.1])[0]
    paiements.append({
        "cart_id": panier["id"],
        "user_id": panier["userId"],
        "amount": panier["discountedTotal"],
        "status": statut,
    })

col = db["raw_payments"]
col.delete_many({})
col.insert_many(paiements)

print("Paiements :", col.count_documents({}))
print("Paniers distincts :", len(col.distinct("cart_id")))

Paiements : 208
Paniers distincts : 208


In [5]:
pipeline = [
    {"$group": {
        "_id": "$status",
        "nombre": {"$sum": 1},
        "montant": {"$sum": "$amount"}
    }},
    {"$sort": {"nombre": -1}}
]

for resultat in db["raw_payments"].aggregate(pipeline):
    print(resultat)

{'_id': 'PAID', 'nombre': 145, 'montant': 2322729.73}
{'_id': 'REFUSED', 'nombre': 42, 'montant': 915022.83}
{'_id': 'PENDING', 'nombre': 21, 'montant': 218957.02}
